In [31]:
### Kapitel 1 Quantensimulator mit Qiskit ####
## Dieses Programm stellt einen Quantenschaltkreis mit einem Hadamard-Gatter und einer Messung dar, der einen Qubit nutzt.
# Nach 1024 Messungen wird die Häufigkeit der Ergebnisse ausgegeben.

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

# 1 Qubit und 1 klassisches Bit
qc = QuantumCircuit(30, 30)

# Hadamard-Gatter bringt Qubit in Überlagerung
qc.h(0)

# Messung des Qubits in das klassische Bit
qc.measure(0, 0)

# Aer Simulator initialisieren
simulator = AerSimulator()

# Circuit kompilieren auf Simulator
compiled_circuit = transpile(qc, simulator)

# Ausführen des Circuits mit 1024 Wiederholungen
job = simulator.run(compiled_circuit, shots=1024)
result = job.result()

counts = result.get_counts()
print(counts)


{'000000000000000000000000000000': 509, '000000000000000000000000000001': 515}


In [5]:
#### Zelle 1: Imports und Datenvorbereitung ####

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from skimage.transform import resize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import NeuralNetworkClassifier
from qiskit_aer import AerSimulator
from qiskit.circuit import ParameterVector

from scipy.optimize import minimize

# Wrapper für scipy SLSQP Optimierer, passend für Qiskit Interface
class ScipySLSQPOptimizer:
    def __init__(self, maxiter=100):
        self.maxiter = maxiter

    def minimize(self, fun, x0, jac=None, bounds=None):
        result = minimize(fun, x0, jac=jac, bounds=bounds, method='SLSQP',
                          options={'maxiter': self.maxiter})
        class ResultWrapper:
            def __init__(self, res):
                self.x = res.x
                self.fun = res.fun
                self.success = res.success
                self.message = res.message
        return ResultWrapper(result)


In [6]:
#### Zelle 2: Datenvorbereitung ####


# MNIST laden und filtern: nur 0 und 1
mnist = fetch_openml('mnist_784', version=1)
X = mnist.data.to_numpy()
y = mnist.target.to_numpy().astype(int)

mask = (y == 0) | (y == 1)
X, y = X[mask], y[mask]

# Normalisieren auf [0,1]
X = X / 255.0

# Runterskalieren auf 2x2 Pixel (4 Features)
X_small = np.array([resize(x.reshape(28,28), (2,2), anti_aliasing=True).flatten() for x in X])

# Trainings- und Testdaten aufteilen
X_train, X_test, y_train, y_test = train_test_split(X_small, y, test_size=0.2, random_state=33)

print(f"Trainingsdaten Form: {X_train.shape}, Testdaten Form: {X_test.shape}")

# Nur die ersten 250 Datenpunkte (200 Train, 50 Test)
X_small_subset = X_small[:250]
y_subset = y[:250]

X_train, X_test, y_train, y_test = train_test_split(
    X_small_subset, y_subset, test_size=0.2, random_state=33
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")



Trainingsdaten Form: (11824, 4), Testdaten Form: (2956, 4)
Train shape: (200, 4), Test shape: (50, 4)


In [7]:
#### Zelle 3: QNN Aufbau ####

num_qubits = 4  # 2x2 Pixel -> 4 Features

backend = AerSimulator()
estimator = Estimator()

# Feature-Map mit genau 16 Parametern (ZZFeatureMap)
feature_map = ZZFeatureMap(num_qubits, reps=1)
print(f"Feature map parameter count: {len(feature_map.parameters)}")  # 16

# Ansatz-Circuit
ansatz = RealAmplitudes(num_qubits, reps=2)

# Neue, eindeutige Parametervektoren für Feature-Map und Ansatz (Namenskonflikte vermeiden)
new_feature_params = ParameterVector('x', len(feature_map.parameters))
new_ansatz_params = ParameterVector('y', len(ansatz.parameters))

feature_map = feature_map.assign_parameters(
    {old: new for old, new in zip(feature_map.parameters, new_feature_params)}, inplace=False)
ansatz = ansatz.assign_parameters(
    {old: new for old, new in zip(ansatz.parameters, new_ansatz_params)}, inplace=False)

circuit = ansatz.compose(feature_map)


Feature map parameter count: 4


C:\Users\juanc\AppData\Local\Temp\ipykernel_22248\3386017727.py:6: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator()


In [8]:
#### Zelle 4: QNN und Klassifikator erstellen ####


qnn = EstimatorQNN(estimator=estimator,
                   circuit=circuit,
                   input_params=new_feature_params,
                   weight_params=new_ansatz_params)

optimizer = ScipySLSQPOptimizer(maxiter=100)

classifier = NeuralNetworkClassifier(neural_network=qnn, optimizer=optimizer)


C:\Users\juanc\AppData\Local\Temp\ipykernel_22248\3480764742.py:4: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = EstimatorQNN(estimator=estimator,


In [9]:
#### Zelle 5: Daten skalieren, trainieren und testen ####


scaler = MinMaxScaler(feature_range=(-1, 1))
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Klassifikator trainieren
classifier.fit(X_train_scaled, y_train)

# Vorhersage auf Testdaten
y_pred = classifier.predict(X_test_scaled)
print("Test Accuracy:", accuracy_score(y_test, y_pred))


Test Accuracy: 0.68


In [12]:
#### Zelle 6: Ergebnisse speichern & plotten ####

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# --- CSV speichern ---
pd.DataFrame(X_train).to_csv("Machine_Leaning_X_train.csv", index=False)
pd.DataFrame(y_train, columns=["label"]).to_csv("Machine_Leaning_y_train.csv", index=False)
pd.DataFrame(X_test).to_csv("Machine_Leaning_X_test.csv", index=False)
pd.DataFrame(y_test, columns=["label"]).to_csv("Machine_Leaning_y_test.csv", index=False)

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0,1])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix (QNN)")
plt.savefig("confusion_matrix.png")
plt.close()

# --- Beispielvorhersagen ---
fig, axes = plt.subplots(2, 5, figsize=(10,5))
for i, ax in enumerate(axes.flat):
    # Bild von 2x2 wieder auf 28x28 vergrößern für bessere Visualisierung
    img = resize(X_test[i].reshape(2,2), (28,28), anti_aliasing=True)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"T:{y_test[i]} P:{y_pred[i]}")
    ax.axis("off")
plt.tight_layout()
plt.savefig("Machine_Learning_sample_predictions.png")
plt.close()
